# EASE sur Google Colab — étape 2, C3

> **Avant un nouveau paquet** : supprimer `results/` et `ease_results.zip` de `Mon Drive/pepite/`.

Référence de précision (D7 de `docs/etape-2-modele.md`), **jamais exportée vers le site**.
Le PC local n'a pas assez de RAM libre (~2 Gio pour ~6 Gio nécessaires), d'où ce notebook.

**Avant de lancer** (sur le PC, depuis `pipeline/`) :

```sh
uv run python colab_bundle.py   # -> data/processed/colab/ease_bundle.zip (~150 Mo)
```

Déposer `ease_bundle.zip` dans **Google Drive → `Mon Drive/pepite/`**.

**Lancer** : Exécution → Tout exécuter (environnement CPU standard suffit, ~12 Go de RAM).
Durée estimée : ~3,5 min par configuration, **~1 h au total** (2 matrices de Gram + 10 inversions 23 019 × 23 019
+ 11 évaluations). L'inversion (`dpotrf` / `dpotri`) tourne sur **un seul thread BLAS** : OpenBLAS multithread
plante (segfault, code 139) sur une matrice de cette taille.
Les résultats sont écrits **au fur et à mesure dans Drive** (`Mon Drive/pepite/results/`) : si la session
est coupée, relancer tout, les configurations déjà faites sont sautées.

**Après** : télécharger `Mon Drive/pepite/ease_results.zip` et le décompresser dans
`pipeline/data/processed/results/` (on obtient `results/ease/*.json` et `results/ease_*_test.json`).

Données MovieLens (usage non commercial) : le Drive doit rester privé ; supprimer le paquet une fois fini.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/pepite"

In [ ]:
import psutil

ram = psutil.virtual_memory()
print(f"RAM totale {ram.total / 2**30:.1f} Gio, disponible {ram.available / 2**30:.1f} Gio")
assert ram.available > 7 * 2**30, "pas assez de RAM : choisir un environnement avec plus de mémoire"

In [ ]:
%%bash
set -e
pip install -q "polars==2.1.0" "python-dotenv==1.2.4"
pip install -q "threadpoolctl==3.8.0"
rm -rf /content/pepite && mkdir -p /content/pepite
cd /content/pepite && unzip -q /content/drive/MyDrive/pepite/ease_bundle.zip
mkdir -p /content/drive/MyDrive/pepite/results
ln -s /content/drive/MyDrive/pepite/results /content/pepite/pipeline/data/processed/results
cat /content/pepite/pipeline/bundle_info.json
python -c "import numpy, scipy, polars, threadpoolctl; print(numpy.__version__, scipy.__version__, polars.__version__, threadpoolctl.__version__)"


In [ ]:
%%bash
cd /content/pepite/pipeline && python train_ease.py ram


## Validation : λ_ease ∈ {100, 300, 1 000, 3 000, 10 000} × {β = 1, binaire}

In [ ]:
%%bash
cd /content/pepite/pipeline && OPENBLAS_NUM_THREADS=1 python -X faulthandler train_ease.py search


In [ ]:
import json
import pathlib

rows = []
for p in sorted(pathlib.Path(f"{DRIVE}/results/ease").glob("ease_*_val.json")):
    r = json.loads(p.read_text())
    m = r["metrics"]
    rows.append(
        (
            m["ndcg@20"]["mean"],
            r["config"]["beta"],
            r["config"]["lambda"],
            m["ndcg@20"]["ci95"],
            m["recall@20"]["mean"],
            m["pop_mediane@20"]["mean"],
            m["couverture@20"],
            r["fit_seconds"],
        )
    )
print("beta  lambda   ndcg@20            recall@20  pop_med   couv    fit_s")
for nd, b, lam, ci, rec, pop, cov, fs in sorted(rows, reverse=True):
    print(f"{b:<5g} {lam:<8g} {nd:.5f} ± {ci:.5f}  {rec:.5f}    {pop:<9.0f} {cov:.4f}  {fs}")

## Test : une seule évaluation, meilleure configuration de validation (NDCG@20)

In [ ]:
%%bash
cd /content/pepite/pipeline && OPENBLAS_NUM_THREADS=1 python -X faulthandler train_ease.py best --split test


In [ ]:
%%bash
cd /content/drive/MyDrive/pepite/results
rm -f ../ease_results.zip
shopt -s nullglob
files=(ease/*.json ease_*_test.json)
if (( ${#files[@]} )); then
  zip -q ../ease_results.zip "${files[@]}" && echo "${#files[@]} fichiers dans ease_results.zip"
else
  echo "aucun résultat dans results/ : pas d'archive créée" >&2
fi
ls -la /content/drive/MyDrive/pepite/
